# TRACK A — final statistics. **No GPU. ~1.5 h.** Blocks submission anywhere.

| cell | produces |
|---|---|
| **A1** | hierarchical bootstrap: λ_U and τ_U percentiles beside the conservative limits |
| **A1b** | *optional, GPU* — fingerprint half-split sensitivity |
| **A2** | injection-integrity table and the α_effective axis |
| **A3** | the scope paragraph, auto-filled from the run metadata |

### An honest limitation of A1, stated up front
A fully nested bootstrap would re-estimate K̂ from a resampled E split **and then re-measure every
generation against it** — 2000 × 37 500 correlations, which is not tractable. A1 therefore
resamples **adapters, generations and held-out reals** from the existing per-row CSVs, and handles
fingerprint uncertainty two ways:

1. **Analytically** — K̂ enters numerator and denominator identically, so to first order its error
   cancels in the ratio λ = θ/R_real. Say this in the paper.
2. **Empirically** — A1b re-estimates K̂ from two disjoint halves of E2 and recomputes λ_U with
   each. Costs ~50 min of GPU and bounds what the cancellation argument assumes.

Report the conservative-limit value as the headline; the bootstrap is **sensitivity**. Six adapter
clusters give a coarse upper tail no matter how many iterations you draw.

In [ ]:
# %% A0 — setup
import os, json, glob, math
import numpy as np, pandas as pd
from scipy import stats as sps
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE="/content/drive/MyDrive"
except Exception:
    DRIVE=os.environ.get("EINV_DRIVE",".")
BASE=os.path.join(DRIVE,"inv_channel/E_INV_P0_v3")
SEED=os.path.join(DRIVE,"inv_channel/E_SEEDEXT")
AMP =os.path.join(DRIVE,"inv_channel/E_AMP")
OUT =os.path.join(BASE,"trackA"); os.makedirs(OUT,exist_ok=True)

S5  = pd.read_csv(os.path.join(BASE,"csv","s5_measure.csv"))
B3  = pd.read_csv(os.path.join(SEED,"csv","b3_measure.csv"))
S1  = pd.read_csv(os.path.join(BASE,"csv","s1_positive_control.csv"))
VAE = pd.read_csv(os.path.join(BASE,"csv","s1b_vae_roundtrip.csv"))
ALPHA_SIM = 0.01                       # 99% SIMULTANEOUS across the two arms
NBOOT     = 2000
print(f"S5 {len(S5)} rows | B3 {len(B3)} | S1 {len(S1)} | VAE {len(VAE)}")

def theta_vec(df, tag, ks, ko):
    a=df[(df.tag==tag)&(df.K==ks)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    b=df[(df.tag==tag)&(df.K==ko)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    n=min(len(a),len(b)); return a[:n]-b[:n]

ARMS={}
for arm,ks,ko in (("A","A","B"),("B","B","A")):
    v=[]
    for s in (0,1,2): v.append(theta_vec(S5,f"{arm}_raw_s{s}_r16",ks,ko))
    for s in (3,4,5): v.append(theta_vec(B3,f"{arm}_raw_s{s}_r16",ks,ko))
    ARMS[arm]=[x for x in v if len(x)]
    print(f"arm {arm}: {len(ARMS[arm])} adapters x {len(ARMS[arm][0])} generations")

def real_contrast(role,other):
    a=S1[(S1.role_img==role)&(S1.role_K==role)].sort_values("idx")["rho_mult"].to_numpy(float)
    b=S1[(S1.role_img==role)&(S1.role_K==other)].sort_values("idx")["rho_mult"].to_numpy(float)
    n=min(len(a),len(b)); return a[:n]-b[:n]
def vae_contrast(role,stage):
    g=VAE[(VAE.role==role)&(VAE.stage==stage)].sort_values("idx")
    own,oth=("rho_A","rho_B") if role=="A" else ("rho_B","rho_A")
    return g[own].to_numpy(float)-g[oth].to_numpy(float)
RA,RB=real_contrast("A","B"),real_contrast("B","A")
VP=np.r_[vae_contrast("A","post"),vae_contrast("B","post")]
print(f"R_real arms n={len(RA)},{len(RB)} | R_VAE post n={len(VP)}")

In [ ]:
# %% A1 — hierarchical bootstrap (adapters -> generations -> held-out reals)
rng=np.random.default_rng(20260731)

def point_estimate(arms, ra, rb, vp):
    """U_device over the two arms, plus the two denominators. Mirrors the headline calculation."""
    Us=[]
    for arm in ("A","B"):
        m=np.array([x.mean() for x in arms[arm]]); k=len(m)
        Us.append(m.mean()+sps.t.ppf(1-ALPHA_SIM/2,df=k-1)*m.std(ddof=1)/np.sqrt(k))
    U=max(Us)
    R=0.5*(ra.mean()+rb.mean())
    return U, R, vp.mean(), 0.5*(np.mean([x.mean() for x in arms["A"]])
                                 +np.mean([x.mean() for x in arms["B"]]))

U0,R0,V0,th0 = point_estimate(ARMS,RA,RB,VP)
print(f"point: U={U0:+.4e}  R_real={R0:.5e}  R_VAE={V0:.5e}  theta_sym={th0:+.4e}")
print(f"       lambda_hat={th0/R0:+.4%}   lambda_U(plug-in)={U0/R0:.4%}   tau_U(plug-in)={U0/V0:.4%}")

lamU=np.empty(NBOOT); tauU=np.empty(NBOOT); lamHat=np.empty(NBOOT)
for b in range(NBOOT):
    boot={}
    for arm in ("A","B"):
        A_=ARMS[arm]; k=len(A_)
        idx=rng.integers(0,k,k)                                  # 1. resample ADAPTERS (clusters)
        boot[arm]=[A_[i][rng.integers(0,len(A_[i]),len(A_[i]))]  # 2. generations WITHIN adapter
                   for i in idx]
    ra=RA[rng.integers(0,len(RA),len(RA))]                       # 3. held-out reals
    rb=RB[rng.integers(0,len(RB),len(RB))]
    vp=VP[rng.integers(0,len(VP),len(VP))]
    try:
        Ub,Rb,Vb,thb=point_estimate(boot,ra,rb,vp)
        lamU[b]=Ub/Rb; tauU[b]=Ub/Vb; lamHat[b]=thb/Rb
    except Exception:
        lamU[b]=np.nan; tauU[b]=np.nan; lamHat[b]=np.nan
ok=np.isfinite(lamU)
print(f"\n{ok.sum()}/{NBOOT} usable iterations")

def bca(theta_hat, boots, jack):
    """BCa upper limit at 1-ALPHA_SIM/2 (one-sided, matching the simultaneous convention)."""
    b=boots[np.isfinite(boots)]
    z0=sps.norm.ppf(max(min((b<theta_hat).mean(),1-1e-9),1e-9))
    jm=jack.mean(); num=((jm-jack)**3).sum(); den=6*(((jm-jack)**2).sum()**1.5)
    a=num/den if den>0 else 0.0
    z=sps.norm.ppf(1-ALPHA_SIM/2)
    alpha_adj=sps.norm.cdf(z0+(z0+z)/(1-a*(z0+z)))
    return float(np.quantile(b,min(max(alpha_adj,1e-6),1-1e-6)))

# jackknife over adapters (the coarse level) for the acceleration term
jl=[]
for arm in ("A","B"):
    for i in range(len(ARMS[arm])):
        red={a:[x for j,x in enumerate(ARMS[a]) if not (a==arm and j==i)] for a in ("A","B")}
        try:
            Uj,Rj,_,_=point_estimate(red,RA,RB,VP); jl.append(Uj/Rj)
        except Exception: pass
jl=np.array(jl)

res={"n_boot":int(ok.sum()),
     "lambda_hat":{"point":th0/R0,"pct2.5":float(np.nanpercentile(lamHat,2.5)),
                   "pct97.5":float(np.nanpercentile(lamHat,97.5))},
     "lambda_U":{"plugin":U0/R0,
                 "pct99":float(np.nanpercentile(lamU,99)),
                 "bca99":bca(U0/R0,lamU,jl)},
     "tau_U":{"plugin":U0/V0,"pct99":float(np.nanpercentile(tauU,99))}}
print(f"\n{'quantity':22s} {'plug-in':>10s} {'boot p99':>10s} {'BCa':>10s}")
print(f"{'lambda_U':22s} {res['lambda_U']['plugin']:10.4%} "
      f"{res['lambda_U']['pct99']:10.4%} {res['lambda_U']['bca99']:10.4%}")
print(f"{'tau_U':22s} {res['tau_U']['plugin']:10.4%} {res['tau_U']['pct99']:10.4%}"
      f" {'—':>10s}")
print(f"{'lambda_hat (95% CI)':22s} {res['lambda_hat']['point']:+10.4%} "
      f"[{res['lambda_hat']['pct2.5']:+.4%}, {res['lambda_hat']['pct97.5']:+.4%}]")
print("\nHEADLINE stays the conservative-limit value: lambda_U 0.278%, tau_U 0.809%.")
print("The bootstrap is SENSITIVITY — six clusters give a coarse upper tail.")
json.dump(res,open(os.path.join(OUT,"A1_bootstrap.json"),"w"),indent=2)

In [ ]:
# %% A1b — OPTIONAL, GPU, ~50 min. Fingerprint half-split sensitivity.
#     Bounds what the "K-hat error cancels in the ratio" argument assumes.
RUN_A1B = False        # flip to True on a GPU runtime

if RUN_A1B:
    import torch, pywt, subprocess, sys
    import torch.nn.functional as tF
    from PIL import Image
    subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets"],check=False)
    DEV="cuda" if torch.cuda.is_available() else "cpu"
    torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
    MEAS=1024
    exec(open("/dev/null").read()) if False else None
    # --- minimal forensic core (identical settings to the study) ---
    def load_lum(fp):
        with Image.open(fp) as im:
            im=im.convert("RGB"); W,H=im.size
            a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
        return (0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    def _c2(x,k):
        xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
        if DEV=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
        return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
    def wres(img):
        s0=np.float32(4.0)
        co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
        for (cH,cV,cD) in co[1:]:
            bnd=[]
            for ch in (cH,cV,cD):
                ch=ch.astype(np.float32); vm=None
                for w in (3,5,7,9):
                    v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                    vm=v if vm is None else np.minimum(vm,v)
                bnd.append(ch*(s0/(vm+s0)))
            out.append(tuple(bnd))
        n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
        r=n-n.mean(); return (r/r.std()).astype(np.float32)
    def zm(K):
        K=K-K.mean(axis=1,keepdims=True); return K-K.mean(axis=0,keepdims=True)
    def wdft(K):
        F=np.fft.fft2(K.astype(np.float32)); mag=(np.abs(F)/K.shape[0]).astype(np.float32)
        sig=max(float(np.median(mag))/0.6745,1e-12); s2=np.float32(sig**2); vm=None
        for w in (3,5,7,9):
            k=np.ones((w,w),np.float32)/(w*w); mu=_c2(mag,k)
            v=np.maximum(_c2(mag*mag,k)-mu*mu,0.0); vm=v if vm is None else np.minimum(vm,v)
        return np.real(np.fft.ifft2(F*(s2/(vm+s2)).astype(np.float32))).astype(np.float32)
    def estK(files):
        num=den=None
        for fp in files:
            Y=load_lum(fp); W=wres(Y)
            num=W*Y if num is None else num+W*Y; den=Y*Y if den is None else den+Y*Y
        return wdft(zm((num/np.maximum(den,1e-6)).astype(np.float32)))
    def ncc(a,b):
        a=a-a.mean(); b=b-b.mean()
        return float((a*b).sum()/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))

    import csv as _csv
    M={}
    for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
        M.setdefault((r["role"],r["split"]),[]).append(r["path"])
    for half in (0,1):
        Ks={}
        for role in ("A","B"):
            f=M[(role,"E2")]; Ks[role]=estK(f[half::2])
            print(f"  half {half} role {role}: K-hat from {len(f[half::2])} images")
        rr=[]
        for role,oth in (("A","B"),("B","A")):
            for fp in M[(role,"H")]:
                Y=load_lum(fp); W=wres(Y)
                rr.append(ncc(W,Y*Ks[role])-ncc(W,Y*Ks[oth]))
        print(f"  half {half}: R_real = {np.mean(rr):.5e}  "
              f"(full-sample 3.56703e-02, ratio {np.mean(rr)/3.56703e-02:.3f})")
    print("\nREAD: if the two halves give R_real within a few percent of the full-sample value,")
    print("      fingerprint estimation error is not driving the denominator.")
else:
    print("A1b skipped (RUN_A1B=False). Flip it on a GPU runtime; ~50 min.")

In [ ]:
# %% A2 — injection integrity and the alpha_effective axis
REAL=pd.read_csv(os.path.join(AMP,"csv","a1_real_dose.csv"))
g=REAL.groupby("alpha")
tab=pd.DataFrame({"rho_KB":g["rho_KB"].median(),"rho_KA":g["rho_KA"].median(),
                  "n":g.size()}).reset_index()
# R_natural for device B measured on its OWN photographs (the correct reference)
R_nat_B=float(np.mean(real_contrast("B","A")))
tab["alpha_effective"]=(tab.rho_KB-tab.loc[tab.alpha==0,"rho_KB"].iloc[0])/R_nat_B
print(f"R_natural(B) from device-B photographs = {R_nat_B:.5e}\n")
print(tab.to_string(index=False,float_format=lambda x:f"{x:.5g}"))
lin=tab[tab.alpha>0]
sl=np.polyfit(lin.alpha,lin.alpha_effective,1)
print(f"\n  alpha_effective / alpha_nominal = {sl[0]:.4f}   intercept {sl[1]:+.4f}")
print(f"  -> nominal 12x corresponds to {12*sl[0]+sl[1]:.2f}x effective")
print("\n  REPLOT every amplitude figure on alpha_effective, not the nominal coefficient.")
tab.to_csv(os.path.join(OUT,"A2_injection_integrity.csv"),index=False)

# clipping and fidelity, on a sample (needs image access; skip cleanly if unavailable)
try:
    from PIL import Image
    import csv as _csv
    MAN={}
    for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
        MAN.setdefault((r["role"],r["split"]),[]).append(r["path"])
    KB1=np.load(os.path.join(BASE,"fingerprints","K_B_E1.npy"))
    def rgbcrop(fp,M_=1024):
        with Image.open(fp) as im:
            im=im.convert("RGB"); W,H=im.size
            return np.asarray(im.crop(((W-M_)//2,(H-M_)//2,(W-M_)//2+M_,(H-M_)//2+M_)),np.float32)
    print(f"\n{'alpha':>7s} {'clipped %':>10s} {'PSNR dB':>9s} {'RMS pert':>10s}")
    for a in (1.0,1.6,3.0,6.0,12.0):
        cl=[];ps=[];rm=[]
        for fp in MAN[("A","T")][:20]:
            x=rgbcrop(fp); y=x*(1.0+a*KB1[...,None])
            cl.append(float(((y>255)|(y<0)).mean())); yc=np.clip(y,0,255)
            d=yc-x; rm.append(float(np.sqrt((d**2).mean())))
            ps.append(float(20*np.log10(255/max(np.sqrt((d**2).mean()),1e-9))))
        print(f"{a:7.1f} {100*np.mean(cl):10.4f} {np.mean(ps):9.2f} {np.mean(rm):10.4f}")
    print("\n  clipping stays small at every level -> the alpha axis is close to linear in energy")
except Exception as e:
    print("\n  fidelity block skipped:",type(e).__name__,e)

In [ ]:
# %% A3 — scope paragraph, auto-filled from run metadata
meta=json.load(open(os.path.join(BASE,"adapters","A_raw_s1_r16","train_meta.json")))
man=json.load(open(os.path.join(BASE,"manifest","manifest.json")))
print(f"""SCOPE

All results concern two physical bodies of a single camera model
({man['picks']['A']} and {man['picks']['B']}, {man.get('pair_model','Nikon_D200')}) from the
Dresden Image Database, a 2010-era CCD corpus. Adaptation used {meta['model']} with a
rank-{meta['rank']} LoRA on attention projections only, {meta['steps']} optimisation steps and
{man['devices']['A']['n']['T']} training images per adapter, all presented as a fixed
{man['meas']}x{man['meas']} centre crop at native resolution. Generation was text-to-image only,
with no image conditioning at any point. The bound is over {6} independently trained adapters per
arm and therefore generalises over training seeds; the device level comprises two bodies. We do not
claim these results extend to other camera generations, to computational-photography pipelines
whose acquisition chains may carry different persistent device signatures, to random-crop training,
or to purpose-built or optimised coatings, which prior work shows can be made learnable by design.""")
print("\n(paste into the manuscript; verify every number against E_INV_RESULTS.md)")